In [1]:
# -*- coding: utf-8 -*-
"""
抖音(CSV) + 微信视频号(XLSX) 订单数据处理脚本
流程：
  1) 分别处理 『抖-*.csv』 和 『视频号-*.xlsx』
  2) 各自输出一个 xlsx（4 达人 sheet + 未匹配）
  3) 合并两源数据，按达人拆 sheet 输出第三个 xlsx（每 sheet 单一表头）
依赖：pip install pandas openpyxl
"""
import os
import glob
import pandas as pd

# ========== 配置区 ==========
FOLDER = r'C:\Users\yuxy15\Desktop\日报\0924\test'

DOUYIN_OUTPUT = os.path.join(FOLDER, '处理结果_抖音.xlsx')
WECHAT_OUTPUT = os.path.join(FOLDER, '处理结果_微信.xlsx')
MERGED_OUTPUT = os.path.join(FOLDER, '处理结果_合并.xlsx')

OUTPUT_COLS = ["达人昵称", "商品数量", "平台", "年级", "商品", "uid", "订单号", "商家备注"]
SHEET_NAMES = ["白瑞芳", "建昆", "胡源", "李珍"]

# 商品名 -> 达人（两平台规则顺序略有不同，各自保留）
PRODUCT_RULES_DY = [("源", "胡源"), ("白", "白瑞芳"), ("建昆", "建昆"), ("珍", "李珍")]
PRODUCT_RULES_WX = [("白", "白瑞芳"), ("建昆", "建昆"), ("珍", "李珍"), ("源", "胡源")]

# 商品属性 -> 年级
GRADE_RULES = [("高一", "新高一"), ("高二", "新高二"), ("高三", "新高三"),
               ("初一", "新初一"), ("初二", "新初二"), ("初三", "新初三")]

# 微信源数据列名 -> 输出列名
WECHAT_RENAME_MAP = {
    "带货账号昵称": "达人昵称",
    # "备注": "商家备注",    # 若微信源里“商家备注”的列名不是“商家备注”，取消注释并按实际改
}


# ========== 工具函数 ==========
def keyword_map(value, rules):
    """按规则顺序做子串包含匹配，命中即止；无命中返回 None"""
    if pd.isna(value):
        return None
    text = str(value)
    for keyword, result in rules:
        if keyword in text:
            return result
    return None


def read_csv_auto(path):
    """自动尝试常见编码读取 CSV（抖音导出通常为 UTF-8-BOM 或 GBK）"""
    for enc in ("utf-8-sig", "utf-8", "gbk"):
        try:
            return pd.read_csv(path, dtype=str, encoding=enc)
        except (UnicodeDecodeError, UnicodeError):
            continue
    raise ValueError(f"无法识别文件编码：{path}")


def ensure_output_cols(df):
    """保证 OUTPUT_COLS 里所有列都存在，缺失的补空"""
    for col in OUTPUT_COLS:
        if col not in df.columns:
            print(f"  [警告] 输出列 {col!r} 在源数据里找不到，将补空")
            df[col] = ""
    return df


# ========== 处理：抖音 CSV ==========
def process_douyin(path):
    df = read_csv_auto(path)

    # 行级过滤
    after_sale = df["售后状态"].astype(str).str.strip()
    df = df[
        (df["订单状态"].astype(str).str.strip() != "已关闭") &
        (after_sale == "-") &
        (df["达人昵称"].notna()) &
        (df["达人昵称"].astype(str).str.strip() != "")
    ].copy()

    # 派生列
    df["商品数量"] = 1
    df["平台"]     = "抖音"
    df["uid"]      = df["达人ID"]
    df["订单号"]   = df["主订单编号"]
    df["商品"]     = df["选购商品"].apply(keyword_map, rules=PRODUCT_RULES_DY)
    df["年级"]     = df["选购商品"].apply(keyword_map, rules=GRADE_RULES)

    return ensure_output_cols(df)


# ========== 处理：微信视频号 XLSX ==========
def process_wechat(path):
    df = pd.read_excel(path, dtype=str)

    # 行级过滤
    daihuo_id = df["带货ID"].astype(str).str.strip()
    df = df[
        (df["订单状态"].astype(str).str.strip() != "已取消") &
        (daihuo_id != "-") & (daihuo_id != "") & (daihuo_id.str.lower() != "nan")
    ].copy()

    # 派生列
    df["商品数量"] = 1
    df["平台"]     = "视频号"
    df["uid"]      = df["带货ID"]
    df["商品"]     = df["商品名称"].apply(keyword_map, rules=PRODUCT_RULES_WX)
    df["年级"]     = df["商品属性"].apply(keyword_map, rules=GRADE_RULES)

    # 列名映射
    df = df.rename(columns=WECHAT_RENAME_MAP)

    return ensure_output_cols(df)


# ========== 输出：按达人拆分（单文件） ==========
def write_split(df, output_file, title):
    unmatched = df[df["商品"].isna()]
    if not unmatched.empty:
        print(f"  [警告] {title} 有 {len(unmatched)} 行未匹配商品，写入『未匹配』")

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for name in SHEET_NAMES:
            part = df[df["商品"] == name]
            part.to_excel(writer, sheet_name=name, index=False, columns=OUTPUT_COLS)
            print(f"  写出『{name}』: {len(part)} 行")
        # 防御：全空时也至少写一个 sheet，避免 openpyxl 报 IndexError
        if not writer.sheets:
            pd.DataFrame(columns=OUTPUT_COLS).to_excel(
                writer, sheet_name="空白", index=False)
        if not unmatched.empty:
            unmatched.to_excel(writer, sheet_name="未匹配", index=False)


# ========== 输出：两源合并（单文件） ==========
def write_merged(df_dy, df_wx, output_file):
    # 只取统一列，纵向拼接；ignore_index 保证不会出现重复表头
    both = pd.concat(
        [df_dy[OUTPUT_COLS], df_wx[OUTPUT_COLS]],
        ignore_index=True,
    )
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for name in SHEET_NAMES:
            part = both[both["商品"] == name]
            part.to_excel(writer, sheet_name=name, index=False)
            print(f"  合并写出『{name}』: {len(part)} 行")
        if not writer.sheets:
            pd.DataFrame(columns=OUTPUT_COLS).to_excel(
                writer, sheet_name="空白", index=False)


# ========== 主流程 ==========
def main():
    if not os.path.isdir(FOLDER):
        raise NotADirectoryError(f"文件夹不存在：{FOLDER}")

    # 1) 找输入文件
    dy_files = sorted(glob.glob(os.path.join(FOLDER, '抖-*.csv')))
    wx_files = sorted(glob.glob(os.path.join(FOLDER, '视频号-*.xlsx')))

    # 2) 处理抖音
    if dy_files:
        print(f"发现 {len(dy_files)} 个抖音文件：")
        for p in dy_files:
            print("   ", p)
        df_dy = pd.concat([process_douyin(p) for p in dy_files], ignore_index=True)
        print(f"抖音合计 {len(df_dy)} 行，写出：{DOUYIN_OUTPUT}")
        write_split(df_dy, DOUYIN_OUTPUT, "抖音")
    else:
        print("未发现『抖-*.csv』文件")
        df_dy = pd.DataFrame(columns=OUTPUT_COLS)

    print()

    # 3) 处理微信
    if wx_files:
        print(f"发现 {len(wx_files)} 个微信文件：")
        for p in wx_files:
            print("   ", p)
        df_wx = pd.concat([process_wechat(p) for p in wx_files], ignore_index=True)
        print(f"微信合计 {len(df_wx)} 行，写出：{WECHAT_OUTPUT}")
        write_split(df_wx, WECHAT_OUTPUT, "微信")
    else:
        print("未发现『视频号-*.xlsx』文件")
        df_wx = pd.DataFrame(columns=OUTPUT_COLS)

    # 4) 合并输出
    print(f"\n合并输出：{MERGED_OUTPUT}")
    write_merged(df_dy, df_wx, MERGED_OUTPUT)

    print("\n全部完成 ✅")


if __name__ == "__main__":
    main()

发现 1 个抖音文件：
    C:\Users\yuxy15\Desktop\日报\0924\test\抖-有道教育专卖.csv
抖音合计 963 行，写出：C:\Users\yuxy15\Desktop\日报\0924\test\处理结果_抖音.xlsx
  写出『白瑞芳』: 99 行
  写出『建昆』: 98 行
  写出『胡源』: 459 行
  写出『李珍』: 307 行

发现 2 个微信文件：
    C:\Users\yuxy15\Desktop\日报\0924\test\视频号-教培小店数据导出.xlsx
    C:\Users\yuxy15\Desktop\日报\0924\test\视频号-有道领世计算机店铺数据导出.xlsx
微信合计 444 行，写出：C:\Users\yuxy15\Desktop\日报\0924\test\处理结果_微信.xlsx
  写出『白瑞芳』: 44 行
  写出『建昆』: 3 行
  写出『胡源』: 283 行
  写出『李珍』: 114 行

合并输出：C:\Users\yuxy15\Desktop\日报\0924\test\处理结果_合并.xlsx
  合并写出『白瑞芳』: 143 行
  合并写出『建昆』: 101 行
  合并写出『胡源』: 742 行
  合并写出『李珍』: 421 行

全部完成 ✅


In [3]:
# -*- coding: utf-8 -*-
"""
抖音(CSV) + 微信视频号(XLSX) 订单数据处理脚本
流程：
  1) 分别处理 『抖-*.csv』 和 『视频号-*.xlsx』（逐文件打印明细统计）
  2) 各自输出一个 xlsx（4 达人 sheet + 未匹配）
  3) 合并两源数据，按达人拆 sheet 输出第三个 xlsx（每 sheet 单一表头）
  4) 所有控制台输出同步写入 txt 日志
依赖：pip install pandas openpyxl
"""
import os
import sys
import glob
import traceback
import pandas as pd
from datetime import datetime

# ========== 配置区 ==========
FOLDER = r'C:\Users\yuxy15\Desktop\日报\0924\test'

DOUYIN_OUTPUT = os.path.join(FOLDER, '处理结果_抖音.xlsx')
WECHAT_OUTPUT = os.path.join(FOLDER, '处理结果_微信.xlsx')
MERGED_OUTPUT = os.path.join(FOLDER, '处理结果_合并.xlsx')

OUTPUT_COLS = ["达人昵称", "商品数量", "平台", "年级", "商品", "uid", "订单号", "商家备注"]
SHEET_NAMES = ["白瑞芳", "建昆", "胡源", "李珍"]

PRODUCT_RULES_DY = [("源", "胡源"), ("白", "白瑞芳"), ("建昆", "建昆"), ("珍", "李珍")]
PRODUCT_RULES_WX = [("白", "白瑞芳"), ("建昆", "建昆"), ("珍", "李珍"), ("源", "胡源")]

GRADE_RULES = [("高一", "新高一"), ("高二", "新高二"), ("高三", "新高三"),
               ("初一", "新初一"), ("初二", "新初二"), ("初三", "新初三")]

WECHAT_RENAME_MAP = {
    "带货账号昵称": "达人昵称",
    # "备注": "商家备注",
}


# ========== 日志工具 ==========
class Tee:
    """同时把 print 内容写到多个流（控制台 + 文件）"""
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            s.write(data)
    def flush(self):
        for s in self.streams:
            s.flush()


# ========== 业务工具 ==========
def keyword_map(value, rules):
    if pd.isna(value):
        return None
    text = str(value)
    for keyword, result in rules:
        if keyword in text:
            return result
    return None


def read_csv_auto(path):
    for enc in ("utf-8-sig", "utf-8", "gbk"):
        try:
            return pd.read_csv(path, dtype=str, encoding=enc)
        except (UnicodeDecodeError, UnicodeError):
            continue
    raise ValueError(f"无法识别文件编码：{path}")


def ensure_output_cols(df):
    for col in OUTPUT_COLS:
        if col not in df.columns:
            print(f"      [警告] 输出列 {col!r} 在源数据里找不到，将补空")
            df[col] = ""
    return df


# ========== 单文件明细统计 ==========
def print_file_stats(path, df, raw_rows, filtered_rows, source_label):
    """打印单个源文件的处理统计"""
    print(f"    ▸ {os.path.basename(path)}")
    print(f"        原始 {raw_rows} 行 → 过滤后 {filtered_rows} 行")

    matched_total = 0
    for name in SHEET_NAMES:
        n = int((df["商品"] == name).sum())
        matched_total += n
        print(f"        {name:<4}: {n} 行")

    n_unmatched = int(df["商品"].isna().sum())
    print(f"        未匹配: {n_unmatched} 行")
    # 交叉核对：分桶合计是否等于过滤后行数
    if matched_total + n_unmatched != filtered_rows:
        print(f"        [核对异常] {matched_total}+{n_unmatched} ≠ {filtered_rows}")


# ========== 处理：抖音 CSV ==========
def process_douyin(path):
    df = read_csv_auto(path)
    raw_rows = len(df)

    after_sale = df["售后状态"].astype(str).str.strip()
    df = df[
        (df["订单状态"].astype(str).str.strip() != "已关闭") &
        (after_sale == "-") &
        (df["达人昵称"].notna()) &
        (df["达人昵称"].astype(str).str.strip() != "")
    ].copy()
    filtered_rows = len(df)

    df["商品数量"] = 1
    df["平台"]     = "抖音"
    df["uid"]      = df["达人ID"]
    df["订单号"]   = df["主订单编号"]
    df["商品"]     = df["选购商品"].apply(keyword_map, rules=PRODUCT_RULES_DY)
    df["年级"]     = df["选购商品"].apply(keyword_map, rules=GRADE_RULES)

    df = ensure_output_cols(df)
    df.attrs['raw_rows'] = raw_rows
    df.attrs['filtered_rows'] = filtered_rows
    return df


# ========== 处理：微信视频号 XLSX ==========
def process_wechat(path):
    df = pd.read_excel(path, dtype=str)
    raw_rows = len(df)

    daihuo_id = df["带货ID"].astype(str).str.strip()
    df = df[
        (df["订单状态"].astype(str).str.strip() != "已取消") &
        (daihuo_id != "-") & (daihuo_id != "") & (daihuo_id.str.lower() != "nan")
    ].copy()
    filtered_rows = len(df)

    df["商品数量"] = 1
    df["平台"]     = "视频号"
    df["uid"]      = df["带货ID"]
    df["商品"]     = df["商品名称"].apply(keyword_map, rules=PRODUCT_RULES_WX)
    df["年级"]     = df["商品属性"].apply(keyword_map, rules=GRADE_RULES)

    df = df.rename(columns=WECHAT_RENAME_MAP)
    df = ensure_output_cols(df)
    df.attrs['raw_rows'] = raw_rows
    df.attrs['filtered_rows'] = filtered_rows
    return df


# ========== 输出：按达人拆分（单文件） ==========
def write_split(df, output_file, title):
    unmatched = df[df["商品"].isna()]
    if not unmatched.empty:
        print(f"      [提示] {title} 共 {len(unmatched)} 行未匹配商品，写入『未匹配』")

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for name in SHEET_NAMES:
            part = df[df["商品"] == name]
            part.to_excel(writer, sheet_name=name, index=False, columns=OUTPUT_COLS)
        if not writer.sheets:
            pd.DataFrame(columns=OUTPUT_COLS).to_excel(
                writer, sheet_name="空白", index=False)
        if not unmatched.empty:
            unmatched.to_excel(writer, sheet_name="未匹配", index=False)


# ========== 输出：两源合并（单文件） ==========
def write_merged(df_dy, df_wx, output_file):
    both = pd.concat(
        [df_dy[OUTPUT_COLS], df_wx[OUTPUT_COLS]],
        ignore_index=True,
    )
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for name in SHEET_NAMES:
            part = both[both["商品"] == name]
            part.to_excel(writer, sheet_name=name, index=False)
        if not writer.sheets:
            pd.DataFrame(columns=OUTPUT_COLS).to_excel(
                writer, sheet_name="空白", index=False)


# ========== 主流程（业务逻辑） ==========
def run_all():
    if not os.path.isdir(FOLDER):
        raise NotADirectoryError(f"文件夹不存在：{FOLDER}")

    print(f"运行时间：{datetime.now():%Y-%m-%d %H:%M:%S}")
    print(f"工作目录：{FOLDER}")
    print("=" * 64)

    dy_files = sorted(glob.glob(os.path.join(FOLDER, '抖-*.csv')))
    wx_files = sorted(glob.glob(os.path.join(FOLDER, '视频号-*.xlsx')))

    # ---------- 抖音 ----------
    print(f"\n【抖音 CSV】共发现 {len(dy_files)} 个文件")
    if dy_files:
        dy_parts = []
        for p in dy_files:
            part = process_douyin(p)
            print_file_stats(
                p, part,
                raw_rows=part.attrs.get('raw_rows'),
                filtered_rows=part.attrs.get('filtered_rows'),
                source_label="抖音",
            )
            dy_parts.append(part)
        df_dy = pd.concat(dy_parts, ignore_index=True)

        print(f"  ── 抖音合计：{len(df_dy)} 行")
        for name in SHEET_NAMES:
            print(f"     {name:<4}: {int((df_dy['商品'] == name).sum())} 行")
        print(f"     未匹配: {int(df_dy['商品'].isna().sum())} 行")
        print(f"  写出：{DOUYIN_OUTPUT}")
        write_split(df_dy, DOUYIN_OUTPUT, "抖音")
    else:
        print("  未发现『抖-*.csv』文件")
        df_dy = pd.DataFrame(columns=OUTPUT_COLS)

    # ---------- 微信 ----------
    print(f"\n【微信视频号 XLSX】共发现 {len(wx_files)} 个文件")
    if wx_files:
        wx_parts = []
        for p in wx_files:
            part = process_wechat(p)
            print_file_stats(
                p, part,
                raw_rows=part.attrs.get('raw_rows'),
                filtered_rows=part.attrs.get('filtered_rows'),
                source_label="微信",
            )
            wx_parts.append(part)
        df_wx = pd.concat(wx_parts, ignore_index=True)

        print(f"  ── 微信合计：{len(df_wx)} 行")
        for name in SHEET_NAMES:
            print(f"     {name:<4}: {int((df_wx['商品'] == name).sum())} 行")
        print(f"     未匹配: {int(df_wx['商品'].isna().sum())} 行")
        print(f"  写出：{WECHAT_OUTPUT}")
        write_split(df_wx, WECHAT_OUTPUT, "微信")
    else:
        print("  未发现『视频号-*.xlsx』文件")
        df_wx = pd.DataFrame(columns=OUTPUT_COLS)

    # ---------- 合并 ----------
    print(f"\n【合并输出】")
    print(f"  写出：{MERGED_OUTPUT}")
    write_merged(df_dy, df_wx, MERGED_OUTPUT)
    both = pd.concat([df_dy[OUTPUT_COLS], df_wx[OUTPUT_COLS]], ignore_index=True)
    for name in SHEET_NAMES:
        print(f"     {name:<4}: {int((both['商品'] == name).sum())} 行")
    print(f"     总计: {len(both)} 行")

    print("\n全部完成 ✅")


# ========== 入口：套一层日志 ==========
def main():
    log_path = os.path.join(FOLDER, f"处理日志_{datetime.now():%Y%m%d_%H%M%S}.txt")
    log_file = open(log_path, "w", encoding="utf-8")
    original_stdout = sys.stdout
    sys.stdout = Tee(original_stdout, log_file)

    try:
        run_all()
    except Exception:
        traceback.print_exc()
        raise
    finally:
        sys.stdout = original_stdout
        log_file.close()
        print(f"\n📄 日志已保存：{log_path}")


if __name__ == "__main__":
    main()

运行时间：2026-09-24 17:06:37
工作目录：C:\Users\yuxy15\Desktop\日报\0924\test

【抖音 CSV】共发现 1 个文件
    ▸ 抖-有道教育专卖.csv
        原始 1119 行 → 过滤后 963 行
        白瑞芳 : 99 行
        建昆  : 98 行
        胡源  : 459 行
        李珍  : 307 行
        未匹配: 0 行
  ── 抖音合计：963 行
     白瑞芳 : 99 行
     建昆  : 98 行
     胡源  : 459 行
     李珍  : 307 行
     未匹配: 0 行
  写出：C:\Users\yuxy15\Desktop\日报\0924\test\处理结果_抖音.xlsx

【微信视频号 XLSX】共发现 2 个文件
    ▸ 视频号-教培小店数据导出.xlsx
        原始 11 行 → 过滤后 3 行
        白瑞芳 : 0 行
        建昆  : 3 行
        胡源  : 0 行
        李珍  : 0 行
        未匹配: 0 行
    ▸ 视频号-有道领世计算机店铺数据导出.xlsx
        原始 538 行 → 过滤后 441 行
        白瑞芳 : 44 行
        建昆  : 0 行
        胡源  : 283 行
        李珍  : 114 行
        未匹配: 0 行
  ── 微信合计：444 行
     白瑞芳 : 44 行
     建昆  : 3 行
     胡源  : 283 行
     李珍  : 114 行
     未匹配: 0 行
  写出：C:\Users\yuxy15\Desktop\日报\0924\test\处理结果_微信.xlsx

【合并输出】
  写出：C:\Users\yuxy15\Desktop\日报\0924\test\处理结果_合并.xlsx
     白瑞芳 : 143 行
     建昆  : 101 行
     胡源  : 742 行
     李珍  : 421 行
     总计: 1407 行

全部完成 ✅
